In [136]:
import httpx
import urllib3
import os
from datetime import datetime

# **Auditoría Pasiva: Cabeceras de Seguridad, Gestión de Sesión y Fingerprinting**

Este notebook expone de manera práctica el impacto de las cabeceras HTTP en la seguridad web. Se desarrolla un motor analítico capaz de evaluar escudos del navegador, directivas de cookies y, adicionalmente, fugas de información estructural (Information Disclosure), consolidando todo en un reporte automatizado en HTML ejecutable para consultorías de Pentesting.

## 1. Fundamentos Teóricos de las Cabeceras de Seguridad

Las cabeceras de respuesta HTTP permiten a un servidor web instruir al navegador sobre cómo comportarse frente a determinados recursos, restringiendo acciones riesgosas e interceptando vectores de ataque comunes antes de que afecten a la aplicación lógica.

### **1.1 Análisis Semántico vs. Validación Sintáctica**
Un error crítico en herramientas automatizadas básicas consiste en validar únicamente la existencia de una cabecera. En un entorno real, una cabecera mal configurada puede ser tan peligrosa como su total ausencia.

Analizaremos las directivas más críticas mediante lógica semántica:

1. **Content-Security-Policy (CSP):** Controla los recursos permitidos en el navegador. Mitiga el Cross-Site Scripting (XSS). Si contiene `'unsafe-inline'`, `'unsafe-eval'` o comodines (`*`), la protección se anula parcialmente, a menos que existan mitigaciones avanzadas como un *Nonce* o *Hash* ($sha256$).
2. **X-Frame-Options (XFO):** Previene ataques de *Clickjacking* al prohibir que el sitio sea embebido en un `<iframe>`. Valores seguros: `DENY` o `SAMEORIGIN`. Puede suplirse mediante la directiva `frame-ancestors` en la CSP.
3. **Strict-Transport-Security (HSTS):** Garantiza comunicaciones exclusivas por HTTPS. Requiere imperativamente el parámetro `max-age=<segundos>`. Sin este, el navegador ignora la directiva, dejando el canal expuesto a ataques de degradación de protocolo (*SSL Stripping*).
4. **X-Content-Type-Options:** Desactiva el *MIME-type sniffing* del navegador, forzándolo a respetar estrictamente el tipo de contenido declarado por el servidor. Su único valor válido es `nosniff`.
5. **Referrer-Policy:** Controla cuánta información sobre la URL de origen se envía a otros sitios cuando el usuario hace clic en un enlace, protegiendo la privacidad de los datos cargados en los parámetros de la URL.

In [137]:
def analyze_security_headers(headers):
    """
    Realiza un análisis semántico profundo sobre los valores de las cabeceras HTTP de seguridad.
    """
    findings = []
    
    # 1. Content-Security-Policy
    if "Content-Security-Policy" in headers:
        csp = headers["Content-Security-Policy"].lower()
        if "unsafe-inline" in csp or "unsafe-eval" in csp or "*" in csp:
            findings.append({
                "element": "Content-Security-Policy",
                "status": "warning",
                "summary": "Configuración Débil",
                "details": "La cabecera está presente pero permite 'unsafe-inline', 'unsafe-eval' o comodines (*). Esto anula gran parte de la protección contra ataques XSS, a menos que se implementen rigurosamente mecanismos de Nonce o Hashes."
            })
        else:
            findings.append({
                "element": "Content-Security-Policy",
                "status": "success",
                "summary": "Correcto",
                "details": "Cabecera presente y configurada con restricciones aparentemente robustas."
            })
    else:
        findings.append({
            "element": "Content-Security-Policy",
            "status": "danger",
            "summary": "FALTA",
            "details": "Ausente. La aplicación está altamente expuesta a ataques XSS (Cross-Site Scripting) e inyección de datos maliciosos en el DOM."
        })

    # 2. X-Frame-Options
    if "X-Frame-Options" in headers:
        xfo = headers["X-Frame-Options"].upper()
        if "DENY" in xfo or "SAMEORIGIN" in xfo:
            findings.append({
                "element": "X-Frame-Options",
                "status": "success",
                "summary": f"Correcto ({xfo})",
                "details": "Protección activa contra ataques de secuestro de clics (Clickjacking)."
            })
        else:
            findings.append({
                "element": "X-Frame-Options",
                "status": "warning",
                "summary": "Configuración Insegura",
                "details": f"Valor configurado: '{xfo}'. Debe utilizarse estrictamente DENY o SAMEORIGIN para ser efectivo."
            })
    else:
        csp = headers.get("Content-Security-Policy", "").lower()
        if "frame-ancestors" in csp:
            findings.append({
                "element": "X-Frame-Options",
                "status": "success",
                "summary": "Mitigado por CSP",
                "details": "La cabecera obsoleta X-Frame-Options no está, pero la directiva moderna 'frame-ancestors' en la CSP suple con éxito su función contra Clickjacking."
            })
        else:
            findings.append({
                "element": "X-Frame-Options",
                "status": "danger",
                "summary": "FALTA",
                "details": "Ausente. El sitio web puede ser embebido en interfaces externas maliciosas, exponiendo a los usuarios a Clickjacking."
            })

    # 3. Strict-Transport-Security (HSTS)
    if "Strict-Transport-Security" in headers:
        hsts = headers["Strict-Transport-Security"].lower()
        if "max-age=" in hsts:
            note = "" if "includesubdomains" in hsts else " (Se recomienda añadir 'includeSubDomains' para proteger subdominios)"
            findings.append({
                "element": "Strict-Transport-Security (HSTS)",
                "status": "success",
                "summary": "Correcto",
                "details": f"Fuerza de manera segura conexiones cifradas HTTPS.{note}"
            })
        else:
            findings.append({
                "element": "Strict-Transport-Security (HSTS)",
                "status": "danger",
                "summary": "Configuración Inválida",
                "details": "Falta el parámetro crítico 'max-age', provocando que los navegadores modernos ignoren por completo la directiva."
            })
    else:
        findings.append({
            "element": "Strict-Transport-Security (HSTS)",
            "status": "danger",
            "summary": "FALTA",
            "details": "Ausente. Permite ataques de degradación de protocolo en la red (SSL Stripping) y vectores Man-in-the-Middle (MitM)."
        })

    # 4. X-Content-Type-Options
    if "X-Content-Type-Options" in headers:
        if "nosniff" in headers["X-Content-Type-Options"].lower():
            findings.append({
                "element": "X-Content-Type-Options",
                "status": "success",
                "summary": "Correcto",
                "details": "Previene de manera efectiva ataques basados en la alteración/adivinación del tipo MIME por parte del navegador."
            })
        else:
            findings.append({
                "element": "X-Content-Type-Options",
                "status": "warning",
                "summary": "Valor Inválido",
                "details": "Debe configurarse estrictamente con el valor 'nosniff' para que sea interpretada adecuadamente."
            })
    else:
        findings.append({
            "element": "X-Content-Type-Options",
            "status": "warning",
            "summary": "FALTA",
            "details": "Ausente. Los navegadores podrían intentar adivinar el tipo de contenido cargado, permitiendo ejecuciones de código malicioso camuflado en archivos estáticos."
        })
        
    # 5. Referrer-Policy
    if "Referrer-Policy" in headers:
        ref = headers["Referrer-Policy"].lower()
        if "unsafe-url" in ref or "no-referrer-when-downgrade" in ref:
            findings.append({
                "element": "Referrer-Policy",
                "status": "warning",
                "summary": "Permisiva",
                "details": f"Configurada como '{headers['Referrer-Policy']}'. Puede llegar a filtrar URLs completas con datos sensibles hacia dominios de terceros. Se sugieren valores como 'strict-origin-when-cross-origin'."
            })
        else:
            findings.append({
                "element": "Referrer-Policy",
                "status": "success",
                "summary": "Correcto",
                "details": f"Configurada con un valor restrictivo seguro: '{headers['Referrer-Policy']}'."
            })
    else:
        findings.append({
            "element": "Referrer-Policy",
            "status": "warning",
            "summary": "FALTA",
            "details": "Ausente. El navegador aplicará el comportamiento por defecto, lo que podría exponer rutas internas de la aplicación en las peticiones salientes hacia enlaces externos."
        })

    return findings

## 2. Gestión de Sesión y Directivas de Cookies

Las cookies representan el mecanismo estándar para mantener el estado de autenticación en la Web. Debido a que el protocolo HTTP es *stateless*, la pérdida o interceptación de una cookie de sesión equivale a la pérdida completa del control de la cuenta del usuario.

### **2.1 Banderas Esenciales de Seguridad**

*   **`Secure`**: Instruye al navegador para que la cookie solo sea transmitida bajo canales cifrados (HTTPS), evitando la fuga en texto plano en redes Wi-Fi públicas.
*   **`HttpOnly`**: Restringe por completo el acceso a la cookie a través de JavaScript (`document.cookie`), sirviendo de contención inmediata ante un ataque de robo de sesión por explotación de XSS.
*   **`SameSite`**: Controla si las cookies se adjuntan automáticamente en solicitudes cruzadas, mitigando ataques de falsificación de peticiones (**CSRF**).

### **2.2 Contextualización de Hallazgos**
Un auditor técnico no trata todas las cookies bajo el mismo estándar. Si una cookie almacena únicamente analíticas o preferencias de idioma (`_octo`, `lang`), la ausencia de `HttpOnly` no representa una vulnerabilidad real, ya que el desarrollo web requiere legítimamente interactuar con ella mediante JavaScript.

In [138]:
def analyze_cookies(cookies):
    """
    Inspecciona las propiedades internas y directivas de seguridad de las cookies capturadas.
    Compatible tanto con objetos de cookies tradicionales como con httpx.jar
    """
    cookie_findings = []
    cookie_objects = cookies.jar if hasattr(cookies, 'jar') else cookies
    
    for cookie in cookie_objects:
        c_name = cookie.name
        
        secure_status = "success" if cookie.secure else "danger"
        secure_desc = "Correcto (HTTPS)" if cookie.secure else "FALTA (Riesgo de interceptación de tráfico)"
        
        # Analizamos las directivas del diccionario interno _rest de forma segura
        is_httponly = 'HttpOnly' in [k for k, _ in cookie._rest.items()] or 'httponly' in [k.lower() for k in cookie._rest.keys()]
        httponly_status = "success" if is_httponly else "danger"
        httponly_desc = "Correcto (Protegida)" if is_httponly else "FALTA (Vulnerable a lectura vía script/XSS)"
        
        samesite = cookie._rest.get('SameSite', cookie._rest.get('samesite', 'No definido'))
        if samesite.lower() in ['lax', 'strict']:
            samesite_status = "success"
            samesite_desc = f"Configurado ({samesite})"
        else:
            samesite_status = "warning"
            samesite_desc = f"Actual: {samesite} (Riesgo potencial de CSRF)"
            
        cookie_findings.append({
            "name": c_name,
            "secure_status": secure_status, "secure_desc": secure_desc,
            "httponly_status": httponly_status, "httponly_desc": httponly_desc,
            "samesite_status": samesite_status, "samesite_desc": samesite_desc
        })
        
    return cookie_findings

## 3. Detección de Fugas de Información (*Fingerprinting*)

En un pentesting profesional, no solo se buscan las cabeceras que faltan para proteger el sitio; también se evalúan las cabeceras que están presentes de forma imprudente. 

Los servidores web mal configurados por defecto tienden a inyectar metadatos informativos sobre las tecnologías subyacentes. Un atacante utiliza esta información para mapear el software del servidor, buscar vulnerabilidades conocidas (*CVEs*) o exploits públicos para dicha versión exacta.

Revisaremos las dos fugas de información más comunes en la industria:
*   **`Server`**: Revela la firma del software del servidor web (ej. `Apache/2.4.41`, `nginx/1.18.0`). Lo ideal en entornos de alta seguridad es que sea totalmente opaca o genérica.
*   **`X-Powered-By`**: Expone el lenguaje de backend o framework que procesa la lógica del negocio (ej. `PHP/7.4.3`, `Express`, `ASP.NET`).

In [139]:
def analyze_fingerprinting(headers):
    """
    Audita las cabeceras de respuesta buscando fugas de información de software o versiones corporativas.
    """
    fingerprint_findings = []
    
    # 1. Evaluación del encabezado 'Server'
    if "Server" in headers:
        server_val = headers["Server"]
        # Si contiene números, es altamente probable que exponga la versión exacta
        if any(char.isdigit() for char in server_val):
            fingerprint_findings.append({
                "element": "Server",
                "status": "danger",
                "value": server_val,
                "recommendation": "Fuga de información crítica. Expone la versión exacta del servidor web. Se recomienda ofuscar o deshabilitar este banner para mitigar ataques dirigidos a exploits conocidos (CVEs)."
            })
        else:
            fingerprint_findings.append({
                "element": "Server",
                "status": "warning",
                "value": server_val,
                "recommendation": "Fuga parcial de información. Expone el software utilizado. Se recomienda enmascarar por completo el valor en producción."
            })
    else:
        fingerprint_findings.append({
            "element": "Server",
            "status": "success",
            "value": "Oculto / Protegido",
            "recommendation": "Correcto. El servidor no expone sus firmas de software subyacentes."
        })
        
    # 2. Evaluación del encabezado 'X-Powered-By'
    if "X-Powered-By" in headers:
        powered_val = headers["X-Powered-By"]
        fingerprint_findings.append({
            "element": "X-Powered-By",
            "status": "danger",
            "value": powered_val,
            "recommendation": "Fuga de información. Revela el lenguaje de programación o backend (ej. PHP, ASP, Node.js). Debe removerse desde la configuración del servidor web o del código de la aplicación."
        })
    else:
        fingerprint_findings.append({
            "element": "X-Powered-By",
            "status": "success",
            "value": "Oculto / Protegido",
            "recommendation": "Correcto. No se detectan tecnologías de backend expuestas de manera pasiva."
        })
        
    return fingerprint_findings

## 4. Reportes Ejecutivos HTML

Función de renderizado encargada de compilar las tres áreas evaluadas e inyectarlas estructuralmente dentro de una plantilla HTML aislada con hojas de estilo embebidas, construyendo el entregable visual de ciberseguridad.

In [140]:
def generate_html_report(url, headers_data, cookies_data, fingerprint_data, tipo_auditoria="Estándar"):
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    clean_url = url.replace('https://','').replace('http://','').replace('/','_').replace(':','_').replace('www.','')
    filename = f"reporte_seguridad_avanzado_{clean_url}.html"
    
    # Configuración de estilos dinámicos según el tipo de resultado
    if tipo_auditoria == "WAF / Bloqueo Activo":
        color_primario = "#991b1b"  # Rojo corporativo (Alerta de bloqueo)
        subtitulo_contexto = "Evaluación de Resiliencia Perimetral (Filtros Activos de Seguridad)"
    elif tipo_auditoria == "Error de Servidor / Anomalía":
        color_primario = "#854d0e"  # Ámbar/Marrón (Comportamiento anómalo)
        subtitulo_contexto = "Análisis de Respuestas Anómalas del Servidor (Posible fuga en Errores)"
    else:
        color_primario = "#1e293b"  # Azul oscuro original
        subtitulo_contexto = "Inspección Semántica Completa y Detección de Fugas de Información"

    # Filas de Cabeceras
    header_rows = ""
    for item in headers_data:
        header_rows += f"""
        <tr>
            <td><strong>{item['element']}</strong></td>
            <td><span class=\"badge badge-{item['status']}\">{item['summary'].upper()}</span></td>
            <td>{item['details']}</td>
        </tr>"""
        
    # Filas de Cookies
    cookie_rows = ""
    if not cookies_data:
        cookie_rows = "<tr><td colspan='4' style='text-align:center; color:#64748b;'>No se capturaron cookies en esta respuesta perimetral.</td></tr>"
    else:
        for cookie in cookies_data:
            cookie_rows += f"""
            <tr>
                <td><code style=\"color:#c7254e; background-color:#f9f2f4; padding:2px 6px; border-radius:4px;\">{cookie['name']}</code></td>
                <td><span class=\"badge badge-{cookie['secure_status']}\">{cookie['secure_desc']}</span></td>
                <td><span class=\"badge badge-{cookie['httponly_status']}\">{cookie['httponly_desc']}</span></td>
                <td><span class=\"badge badge-{cookie['samesite_status']}\">{cookie['samesite_desc']}</span></td>
            </tr>"""
            
    # Filas de Fingerprinting
    fingerprint_rows = ""
    for f_item in fingerprint_data:
        fingerprint_rows += f"""
        <tr>
            <td><strong>{f_item['element']}</strong></td>
            <td><span class=\"badge badge-{f_item['status']}\">{f_item['value']}</span></td>
            <td>{f_item['recommendation']}</td>
        </tr>"""

    html_template = f"""<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <title>Reporte de Reconocimiento Pasivo: {tipo_auditoria}</title>
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f4f6f9; color: #333; margin: 0; padding: 40px; }}
        .container {{ max-width: 1000px; background: white; margin: auto; padding: 30px; border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); }}
        h1 {{ color: {color_primario}; border-bottom: 2px solid #e2e8f0; padding-bottom: 15px; font-size: 26px; }}
        h2 {{ color: #334155; margin-top: 40px; font-size: 18px; border-left: 4px solid {color_primario}; padding-left: 10px; }}
        .meta {{ background: #f8fafc; padding: 15px; border-radius: 6px; margin-bottom: 30px; font-size: 14px; color: #64748b; }}
        table {{ width: 100%; border-collapse: collapse; margin-top: 15px; font-size: 14px; }}
        th, td {{ padding: 12px 15px; text-align: left; border-bottom: 1px solid #e2e8f0; }}
        th {{ background-color: #f1f5f9; color: #475569; font-weight: 600; }}
        .badge {{ padding: 6px 12px; border-radius: 50px; font-size: 11px; font-weight: bold; display: inline-block; }}
        .badge-success {{ background-color: #dcfce7; color: #15803d; }}
        .badge-warning {{ background-color: #fef9c3; color: #a16207; }}
        .badge-danger {{ background-color: #fee2e2; color: #b91c1c; }}
        footer {{ margin-top: 50px; text-align: center; font-size: 12px; color: #94a3b8; border-top: 1px solid #e2e8f0; padding-top: 20px; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>Informe de Reconocimiento Pasivo y Análisis Semántico Perimetral</h1>
        <div class="meta">
            <strong>Objetivo Evaluado:</strong> {url}<br>
            <strong>Fecha de Análisis:</strong> {now}<br>
            <strong>Estado / Tipo de Contexto:</strong> {tipo_auditoria}<br>
            <strong>Alcance Técnico:</strong> {subtitulo_contexto}
        </div>

        <h2>1. Análisis de Cabeceras HTTP Expuestas</h2>
        <table>
            <thead><tr><th>Cabecera Analizada</th><th>Estado</th><th>Detalles Técnicos y Diagnóstico</th></tr></thead>
            <tbody>{header_rows}</tbody>
        </table>

        <h2>2. Auditoría de Parámetros en Cookies de Respuesta</h2>
        <table>
            <thead><tr><th>Identificador Cookie</th><th>Secure</th><th>HttpOnly</th><th>SameSite</th></tr></thead>
            <tbody>{cookie_rows}</tbody>
        </table>

        <h2>3. Detección de Fugas de Información (Fingerprinting)</h2>
        <table>
            <thead><tr><th>Componente Evaluado</th><th>Firma Detectada</th><th>Recomendación de Remediación</th></tr></thead>
            <tbody>{fingerprint_rows}</tbody>
        </table>

        <footer>Módulo de Reconocimiento Pasivo y Resiliencia Perimetral - Toolkit Profesional de Pentesting.</footer>
    </div>
</body>
</html>
"""
    with open(filename, "w", encoding="utf-8") as f:
        f.write(html_template)
    return filename

## 5. Demostración de Ejecución Completa

In [141]:
def ejecutar_auditoria(target_url):
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    if not target_url.startswith(("http://", "https://")):
        target_url = "https://" + target_url
        
    custom_headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
        "Accept-Language": "es-419,es;q=0.9,en;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1"
    }
    
    print(f"[*] Iniciando auditoría perimetral en: {target_url}\n")

    try:
        with httpx.Client(http2=True, timeout=12.0, verify=False) as client:
            respuesta = client.get(target_url, headers=custom_headers)
        
        # Procesamos análisis base sobre lo que sea que nos devuelva el servidor
        resultados_headers = analyze_security_headers(respuesta.headers)
        resultados_cookies = analyze_cookies(respuesta.cookies)
        resultados_fingerprint = analyze_fingerprinting(respuesta.headers)

        # CASO 1: EL SITIO ESTÁ ABIERTO (200 OK) -> Auditoría Estándar
        if respuesta.status_code == 200:
            print(f"[✓] Conexión exitosa (200 OK). Procesando artefactos...")
            print("[🔄] Generando reporte final de auditoría...")
            
            reporte = generate_html_report(target_url, resultados_headers, resultados_cookies, resultados_fingerprint, tipo_auditoria="Auditoría Estándar")
            print(f"[+] Reporte ejecutivo exportado en: {os.path.abspath(reporte)}")

        # CASO 2: EL SITIO TIENE ESCUDO (403/406/999) -> Reporte Especial de WAF
        elif respuesta.status_code in [403, 406, 999]:
            print(f"[!] Acceso restringido por el servidor (Código HTTP {respuesta.status_code}).")
            
            servidor = respuesta.headers.get("Server", "Oculto/No especificado")
            es_akamai = "SÍ" if "X-Akamai-Transformed" in respuesta.headers or "akamai" in servidor.lower() else "No evidente"
            es_cloudflare = "SÍ" if "cf-ray" in respuesta.headers or "cloudflare" in servidor.lower() else "No evidente"
            
            resultados_fingerprint.insert(0, {
                "element": "Protección Perimetral Activa (WAF)",
                "status": "danger",
                "value": f"Código {respuesta.status_code} Detectado",
                "recommendation": f"Se interceptó la petición. Las firmas analizadas pertenecen al escudo perimetral (Cloudflare: {es_cloudflare} | Akamai: {es_akamai}). Se recomienda auditar la política de bloqueo ante evasiones avanzadas."
            })
            
            print("[*] Generando reporte de resiliencia perimetral...")
            reporte = generate_html_report(target_url, resultados_headers, resultados_cookies, resultados_fingerprint, tipo_auditoria="WAF / Bloqueo Activo")
            print(f"[+] Reporte de bloqueo exportado en: {os.path.abspath(reporte)}")
            
            # Resumen técnico limpio en la consola
            print("\n" + "="*65)
            print("  INFORME EN CONSOLA: EVALUACIÓN DE RESILIENCIA PERIMETRAL (WAF) ")
            print("=" * 65)
            print(f"Sitio Web Analizado   : {target_url}")
            print(f"Firma de Servidor (WAF): {servidor}")
            print(f"Detección Akamai      : {es_akamai}")
            print(f"Detección Cloudflare  : {es_cloudflare}")
            print("=" * 65)

        # CASO 3: CUALQUIER OTRO CÓDIGO (Errores 500, 404, etc.)
        else:
            print(f"[?] Código de estado anómalo detectado: {respuesta.status_code}")
            
            resultados_fingerprint.insert(0, {
                "element": "Respuesta Anómala del Servidor",
                "status": "warning",
                "value": f"Código {respuesta.status_code}",
                "recommendation": "El servidor respondió con un estado fuera de lo común. Validar si el cuerpo del error expone stack traces, rutas absolutas o fugas de código interno."
            })
            
            print("[🔄] Generando reporte de anomalía...")
            reporte = generate_html_report(target_url, resultados_headers, resultados_cookies, resultados_fingerprint, tipo_auditoria="Error de Servidor / Anomalía")
            print(f"[+] Reporte anómalo exportado en: {os.path.abspath(reporte)}")

    # CASO 4: EL SITIO ESTÁ COMPLETAMENTE CAÍDO (No hay respuesta HTTP)
    except httpx.RequestError as error_red:
        print(f"[X] Fallo crítico de red: El host no responde o el dominio no existe.")
        print(f"    Detalles técnicos: {error_red}")
    
URL = ""
ejecutar_auditoria(URL)

[*] Iniciando auditoría perimetral en: https://

[X] Fallo crítico de red: El host no responde o el dominio no existe.
    Detalles técnicos: Request URL is missing an 'http://' or 'https://' protocol.
